# Chapter 03: Intelligent Agents & AI Problem Solving (Companion Notebook)

**AI Learning Hub** — Institut Teknologi Sumatera  
Pengampu: Aji Pamungkas Tri Nurcahyo  

Notebook ini merupakan materi pendamping praktik untuk memelajari implementasi agen cerdas: Simple Reflex, Model-Based, Kerangka Kerja PEAS, Ruang Keadaan Grid, dan Formulasi Masalah Pencarian formal.

## 1. Simple Reflex Agent (Pengatur Suhu Sederhana)
Agen refleks sederhana hanya memetakan persepsi saat ini ke tindakan diskret berdasarkan aturan kondisi-aksi (*condition-action rules*) tanpa mengingat status masa lalu.

In [1]:
class SimpleReflexTemperatureAgent:
    def __init__(self, target_temp=24.0, tolerance=1.0):
        self.target_temp = target_temp
        self.tolerance = tolerance

    def select_action(self, percept):
        temp = percept.get("temperature", 24.0)
        occupied = percept.get("occupied", False)
        if not occupied:
            return "ECO_STANDBY"
        if temp > self.target_temp + self.tolerance:
            return "COOLING_MAX"
        elif temp < self.target_temp - self.tolerance:
            return "HEATING_MODERATE"
        return "FAN_IDLE"

agent = SimpleReflexTemperatureAgent()
test_percepts = [
    {"temperature": 28.5, "occupied": True},
    {"temperature": 24.2, "occupied": True},
    {"temperature": 28.5, "occupied": False}
]
for p in test_percepts:
    print(f"Percept: {p} -> Tindakan: {agent.select_action(p)}")

## 2. Model-Based Reflex Agent (Internal State & Anti-Chattering)
Agen berbasis model memelihara status internal untuk melacak tren perubahan suhu dan mencegah osilasi sakelar (*chattering*).

In [2]:
class ModelBasedRoomAgent:
    def __init__(self, target_temp=24.0, tolerance=1.0):
        self.target_temp = target_temp
        self.tolerance = tolerance
        self.previous_temp = None
        self.last_action = "OFF"

    def update_trend(self, current_temp):
        if self.previous_temp is None:
            trend = "STABLE"
        elif current_temp > self.previous_temp + 0.2:
            trend = "RISING"
        elif current_temp < self.previous_temp - 0.2:
            trend = "FALLING"
        else:
            trend = "STABLE"
        self.previous_temp = current_temp
        return trend

    def select_action(self, percept):
        temp = percept.get("temperature", 24.0)
        trend = self.update_trend(temp)
        if temp > self.target_temp + self.tolerance:
            act = "COOLING_HIGH"
        elif temp < self.target_temp - self.tolerance:
            act = "HEATING_MILD"
        else:
            act = "COOLING_LOW" if trend == "FALLING" else "COMFORT_MAINTAIN"
        self.last_action = act
        return f"{act} (Tren: {trend})"

mb_agent = ModelBasedRoomAgent()
stream = [28.0, 26.5, 24.8, 24.0]
for s in stream:
    print(f"Suhu: {s}°C -> Keputusan: {mb_agent.select_action({'temperature': s, 'occupied': True})}")

## 3. PEAS Framework & Evaluasi Kinerja Rasional
Spesifikasi kuantitatif untuk Performance Measure, Environment, Actuators, dan Sensors.

In [3]:
class Evaluator:
    def __init__(self, target_temp=24.0):
        self.target_temp = target_temp
        self.energy_kwh = 0.0
        self.discomfort = 0.0

    def step(self, temp, occupied, action):
        costs = {"COOLING_MAX": 2.5, "COOLING_LOW": 1.0, "ECO_STANDBY": 0.05}
        self.energy_kwh += costs.get(action, 0.5)
        if occupied and abs(temp - self.target_temp) > 1.5:
            self.discomfort += (abs(temp - self.target_temp) - 1.5) * 5.0

    def score(self):
        return max(0.0, 100.0 - (self.energy_kwh * 2.0 + self.discomfort * 3.0))

evaluator = Evaluator()
evaluator.step(28.0, True, "COOLING_MAX")
evaluator.step(24.5, True, "COOLING_LOW")
evaluator.step(24.0, False, "ECO_STANDBY")
print(f"Total Energi: {evaluator.energy_kwh:.2f} kWh, Penalti Discomfort: {evaluator.discomfort:.2f}")
print(f"Skor Rasionalitas Akhir: {evaluator.score():.2f} / 100.00")

## 4. Ruang Keadaan Grid (State Space Representation)
Mendefinisikan himpunan keadaan diskret, aksi legal, dan model transisi deterministik.

In [4]:
class GridSpace:
    def __init__(self, rows=3, cols=3, obstacles=None):
        self.rows, self.cols = rows, cols
        self.obstacles = obstacles or set()
        self.vectors = {"UP": (-1, 0), "DOWN": (1, 0), "LEFT": (0, -1), "RIGHT": (0, 1)}

    def is_valid(self, s):
        r, c = s
        return 0 <= r < self.rows and 0 <= c < self.cols and (r, c) not in self.obstacles

    def actions(self, s):
        r, c = s
        return [act for act, (dr, dc) in self.vectors.items() if self.is_valid((r + dr, c + dc))]

    def transition(self, s, act):
        if act not in self.actions(s): return s
        dr, dc = self.vectors[act]
        return (s[0] + dr, s[1] + dc)

grid = GridSpace(rows=3, cols=3, obstacles={(1, 1)})
print("Aksi legal di (0,0):", grid.actions((0, 0)))
print("Transisi (0,0) + DOWN:", grid.transition((0, 0), "DOWN"))

## 5. Formulasi Masalah Pencarian Formal (Problem Formulation)
Mendefinisikan 5 komponen baku Russell & Norvig: Initial State, Actions, Result, Goal Test, Path Cost.

In [5]:
class SearchProblem:
    def __init__(self, init, goal, grid_env):
        self.init = init
        self.goal = goal
        self.env = grid_env

    def actions(self, s): return self.env.actions(s)
    def result(self, s, a): return self.env.transition(s, a)
    def is_goal(self, s): return s == self.goal
    def step_cost(self, s, a, ns): return 1.0

problem = SearchProblem((0, 0), (2, 2), grid)
plan = ["DOWN", "DOWN", "RIGHT", "RIGHT"]
curr = problem.init
total_cost = 0.0
for act in plan:
    nxt = problem.result(curr, act)
    total_cost += problem.step_cost(curr, act, nxt)
    curr = nxt
print(f"Posisi Akhir: {curr}, Goal Reached: {problem.is_goal(curr)}, Biaya Jalur: {total_cost}")